# Data Preprocess

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

from config import FPS_GAME_IDS

DATA_DIR = Path("data/processed")
OUTPUT_DIR = Path("data/merged")
GENRE = "FPS"


In [2]:
def load_reviews(data_dir, game_ids, genre):
    frames = []

    for game, appid in game_ids.items():
        path = data_dir / f"reviews_{game}_{appid}.csv"

        df = pd.read_csv(
            path,
            dtype={
                "recommendationid": "string",
                "review": "string",
                "voted_up": "boolean",
            },
        )

        df["game"] = game
        df["appid"] = appid
        df["genre"] = genre
        frames.append(df)

    return pd.concat(frames, ignore_index=True)

In [3]:
def preprocess_reviews(df):
    df = df.copy()

    # drop empty rows
    df = df.dropna(
        subset=["recommendationid", "review", "voted_up"]
    )

    # drop duplicate reviews
    df = df.drop_duplicates(
        subset=["appid", "recommendationid"]
    )

    # recommend: 1，not recommend: 0
    df["voted_up"] = df["voted_up"].astype("int8")

    return df.reset_index(drop=True)

In [4]:
def inspect_reviews(df):
    # Inspect data without changing the DataFrame.
    print(f"Total reviews: {len(df)}")
    display(df.head())

    print("Missing values:")
    display(df.isna().sum().to_frame("missing_count"))

    text = (
        df["review"]
        .fillna("")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    print("Empty or whitespace-only reviews:", int(text.eq("").sum()))
    print(
        "Duplicate review IDs:",
        int(df.duplicated(["appid", "recommendationid"]).sum()),
    )
    print(
        "Repeated text beyond the first occurrence (ignoring whitespace differences):",
        int(text[text.ne("")].duplicated().sum()),
    )

    print("Label distribution (0 = not recommended, 1 = recommended):")
    counts = df["voted_up"].value_counts().reindex([0, 1], fill_value=0)
    display(pd.DataFrame({
        "count": counts,
        "proportion": (counts / max(len(df), 1)).round(4),
    }))

    print("Label counts by game:")
    display(
        pd.crosstab(df["game"], df["voted_up"])
        .reindex(columns=[0, 1], fill_value=0)
        .rename(columns={0: "not_recommended", 1: "recommended"})
    )


In [5]:
reviews = load_reviews(DATA_DIR, FPS_GAME_IDS, GENRE)
reviews = preprocess_reviews(reviews)

inspect_reviews(reviews)

Total reviews: 20172
Missing values:
Empty or whitespace-only reviews: 0
Duplicate review IDs: 0
Repeated text beyond the first occurrence (ignoring whitespace differences): 710
Label distribution (0 = not recommended, 1 = recommended):
Label counts by game:


,recommendationid,review,voted_up,language,timestamp_created,timestamp_updated,playtime_at_review,game,appid,genre
0,239306808,fix your useless fucking anticheat which is ba...,0,english,2026-11-03 21:56:37,2026-11-03 22:05:54,16343.0,CS2,730,FPS
1,238765240,"Cheats everywhere, maybe do something about it...",0,english,2026-10-28 18:50:59,2026-10-28 18:50:59,181830.0,CS2,730,FPS
2,238777093,gay idiot stupid face game,0,english,2026-10-28 22:23:34,2026-10-28 22:23:34,4169.0,CS2,730,FPS
3,238458264,i am not cheater,0,english,2026-10-24 14:02:41,2026-10-24 14:02:41,497.0,CS2,730,FPS
4,239016472,I got banned wihtout cheating,0,english,2026-10-31 14:53:16,2026-10-31 14:53:16,578.0,CS2,730,FPS


,missing_count
recommendationid,0
review,0
voted_up,0
language,0
timestamp_created,0
timestamp_updated,0
playtime_at_review,3363
game,0
appid,0
genre,0


,count,proportion
voted_up,,
0,11341,0.5622
1,8831,0.4378


voted_up,not_recommended,recommended
game,,
APEX,1489,1102
BATTLEFIELD_6,1778,1279
COD_MW4,1676,1688
CS2,1431,933
DELTA_FORCE,1693,1187
RAINBOW_SIX_SIEGE,1495,1094
TITANFALL_2,1779,1548


In [6]:
# split train data based on game and voted_up
reviews["stratify_label"] = (
    reviews["game"].astype(str) + "_" +
    reviews["voted_up"].astype(str)
)

train_df,temp_df = train_test_split(
    reviews, test_size=0.2, stratify=reviews["stratify_label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["stratify_label"],
    random_state=42
)

# Save datasets
train_df.to_csv(OUTPUT_DIR / f"{GENRE}_train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / f"{GENRE}_val.csv", index=False)
test_df.to_csv(OUTPUT_DIR / f"{GENRE}_test.csv", index=False)

print(f"Train: {len(train_df)} reviews")
print(f"Validation: {len(val_df)} reviews")
print(f"Test: {len(test_df)} reviews")

Train: 16137 reviews
Validation: 2017 reviews
Test: 2018 reviews


In [7]:
# save whole processed dataframe
output_path = OUTPUT_DIR / f"{GENRE}_reviews_merged.csv"

reviews.to_csv(output_path, index=False, encoding="utf-8")

print(f"Saved {len(reviews)} reviews to: {output_path}")

Saved 20172 reviews to: data\merged\FPS_reviews_merged.csv
